<h1 style="font-size: 36px; color: blue;">STOWA Proevenverzameling tool v5.0</h1>

## Benodigde installaties

In [1]:
# Importeren van benodigde package
# Na het eerte keer installeren van de benodigde packages de kernel handmatig opnieuw optarten via het menu Kernel → Restart

path_to_wheel = r"C:\Users\deenekat7271\Documents\GitHub\pv-tool\dist\pv_tool-0.2.7-py3-none-any.whl"
!pip install "{path_to_wheel}"

Defaulting to user installation because normal site-packages is not writeable
Processing c:\users\deenekat7271\documents\github\pv-tool\dist\pv_tool-0.2.7-py3-none-any.whl
pv-tool is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.


In [2]:
# all imports
import importlib.util
import ipywidgets as widgets
from IPython.display import display, Markdown
from ipyfilechooser import FileChooser
from pathlib import Path
import os

from pv_tool.imports.import_data import Dbase
from pv_tool.cphi_analysis.c_phi_analysis import CPhiAnalyse
from pv_tool.cphi_analysis.variables import *

# widgetfuncties nog verder aanvullen!
from widgetfunctie_cphi import *

In [3]:
# Controleert en installeer indien nodig
packages = ['openpyxl', 'ipyfilechooser', 'pandas_schema', 'xlsxwriter', 'ipywidgets', 'reportlab', 'reportlab', 'kaleido']
for package in packages:
    check_package_install(package)

**openpyxl is al geïnstalleerd.**

**ipyfilechooser is al geïnstalleerd.**

**pandas_schema is al geïnstalleerd.**

**xlsxwriter is al geïnstalleerd.**

**ipywidgets is al geïnstalleerd.**

**reportlab is al geïnstalleerd.**

**reportlab is al geïnstalleerd.**

**kaleido is al geïnstalleerd.**

## Stap 1: Inladen van benodigde data

In [4]:
# Data inladen
dropdown_template = create_import_dropdown()
file_chooser_import = create_file_chooser()
display(Markdown("**Stap 1: Kies Excel template voor uploaden data:**"))
display(dropdown_template)
display(file_chooser_import)

**Stap 1: Kies Excel template voor uploaden data:**

Dropdown(description='Template:', index=2, layout=Layout(width='400px'), options=('Proevenverzamelingtool 4.2n…

FileChooser(path='C:\', filename='', title='Selecteer een bestand om te uploaden', show_hidden=False, select_d…

## Selecteer export locatie

In [5]:
dir_chooser, name_box = select_export_location_and_name()

## Stap 2a: Importeren en valideren data (inclusief toevoegen/herberekenen analyse kolommen)

In [6]:
import time

In [9]:
timestart = time.time()
dbase = Dbase()

if not dropdown_template.value:
    print("Er is geen template geselecteerd.")
elif not file_chooser_import.selected:
    print("Er is geen bestand geselecteerd. Selecteer een bestand voordat je verder gaat.")
elif not dir_chooser.selected_path:
    print("Selecteer een exportlocatie.")
else:
    chosen_filename = name_box.value if name_box.value else None
    process_import_and_validate(
        dbase=dbase,
        template_name=dropdown_template.value,
        file_path=file_chooser_import.selected,
        export_dir=Path(dir_chooser.selected_path)
    )
print(f"Duur: {time.time() - timestart:.2f} seconden")

**Template-code:** PV-tool

number of critical_errors in validate_kenmerken_boring = 0
number of critical_errors in validate_clas = 0
number of critical_errors in validate_crs = 4
number of critical_errors in validate_samendrukking = 0
number of critical_errors in validate_monster = 0
number of critical_errors in validate_triaxiaal = 28
number of critical_errors in validate_dss = 17
number of warnings in validate_alg = 0
number of warnings in validate_kenmerken_boring = 353
number of warnings in validate_clas = 1843
number of warnings in validate_crs = 76
number of warnings in validate_samendrukking = 246
number of warnings in validate_monster = 0
number of warnings in validate_triaxiaal = 96
number of warnings in validate_dss = 33


**Validatierapporten geëxporteerd naar:** C:\Users\deenekat7271\Documents\GitHub\pv-tool

Duur: 44.46 seconden


## Stap 2b: Als errors zijn opgelost exporteer naar Template

In [10]:
# Export dbase-template
timestart = time.time()
process_export(
    dbase,
    export_dir=Path(dir_chooser.selected_path),
    filename=name_box.value if name_box.value else None
)
print(f"Duur: {time.time() - timestart:.2f} seconden")

DataFrame naar template geëxporteerd


**DataFrame geëxporteerd naar:** `C:\Users\deenekat7271\Documents\GitHub\pv-tool\test4.xlsx`

Duur: 21.76 seconden


## Bepalen gedraineerde parameters triaxiaalproeven en DSS-proeven op basis van fit (C en Phi)

In [11]:
# Dataframe met proefdata
dbase_df = dbase.dbase_df

# Lijsten maken
PV_txt_lijst, PV_dss_lijst = maak_verzamelings_lijsten(dbase_df)

# Widgets aanmaken
(dropdown_type_proef, dropdown_rekpercentage_txt, dropdown_rekpercentage_dss,
 dropdown_verzameling, multi_select_verzameling, container_rekpercentage, output_rekpercentage) = maak_proef_widgets(PV_txt_lijst, PV_dss_lijst)

# 'gekozen_rekpercentage' als lijst zodat het binnen callbacks aanpasbaar blijft
gekozen_rekpercentage = ['eindsterkte']

# Koppel callbacks
koppel_callbacks(
    dropdown_type_proef, dropdown_rekpercentage_txt, dropdown_rekpercentage_dss,
    dropdown_verzameling, multi_select_verzameling, container_rekpercentage, output_rekpercentage,
    PV_txt_lijst, PV_dss_lijst, gekozen_rekpercentage
)

# Toon alles
toon_widgets(
    dropdown_type_proef, dropdown_verzameling, container_rekpercentage, output_rekpercentage, multi_select_verzameling
)

**Kies type proef:**

Dropdown(description='Type proef:', layout=Layout(width='400px'), options=('TXT_CPhi', 'DSS_CPhi'), value='TXT…

**Kies verzameling voor statistische analyse:**

Dropdown(description='Verzameling:', layout=Layout(width='400px'), options=('geen', 'TXT_testset_klei', 'TXT_W…

**Kies rekpercentage s'en t:**

Output()

Output()

**Kies één of meerdere verzamelingen om naast de gekozen verzameling voor de statistische analyse te tonen:**

SelectMultiple(description='Vergelijk met:', index=(0,), layout=Layout(height='150px', width='400px'), options…

In [13]:
analyse, coh_gem, phi_kar, coh_kar, partphi, partcoh, typeverzameling = voer_cphi_analyse_uit(
    dbase,
    dropdown_verzameling,
    dropdown_type_proef,
    dropdown_rekpercentage_txt,
    dropdown_rekpercentage_dss,
    dir_chooser,      # FileChooser uit select_export_location_and_name
    name_box,         # Text widget uit select_export_location_and_name
    gekozen_rekpercentage,
    toon_cphi_tabel
)

C:\Users\deenekat7271\Documents\GitHub\pv-tool\test4.xlsx
Er zijn geen eerdere resultaten gevonden voor de opgegeven parameters.


C:\Users\deenekat7271\Documents\GitHub\pv-tool\pv_tool\cphi_analysis\calc_parameters.py:214: UserWarning:

Om berekening standaarddeviatie voor D-stability te kunnen doen moet rekenwaarde van de cohesie waarde positief zijn, gevonden waarde: -1.276. Rekenwaarde cohesie wordt aangepast naar 0.01 om berekening standaarddeviatie voor D-stability te kunnen doen



**Stel de raaklijnen voor de gemiddelde en karakteristieke waarden van de cohesie en hoek van inwendige wrijving vast:**

Op basis van regressie wordt een eerste benadering gegeven voor het snijpunt met de y-as (a1) en de helling (a2) voor de gemiddelde en karakteristieke waarde.

Toets in de volgende stap bij het genereren van de grafieken of de raaklijnen juist zijn gekozen en pas deze aan naar eigen inzicht

De invoer wordt automatisch opgehaald uit het template_PVtool5_0.xlsx [resultaten] indien er eerder resultaten zijn opgeslagen voor de betreffende verzameling.

**Verzameling en rekpercentage: TXT_SAFE_klei_licht_16_175, s'-t bij: ['eindsterkte']**

**Opgeven invoer effectieve schuifsterkteparameters (fit):**

GridspecLayout(children=(Label(value='Beschrijving', layout=Layout(grid_area='widget001')), Label(value='Benad…